# Gold: hospital_daily_summary

Builds `<catalog>.gold.hospital_daily_summary` per the Silver-to-Gold STTM: one row per hospital per day with lab test activity, abnormal/critical rates and a day-over-day trend (LAG). Full rebuild (CREATE OR REPLACE TABLE AS SELECT).

In [0]:
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")

In [0]:
%sql
USE CATALOG IDENTIFIER(:catalog);

CREATE OR REPLACE TABLE gold.hospital_daily_summary
COMMENT 'One row per hospital per day: lab test volume, patients, doctors, abnormal/critical rates and day-over-day trend'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                                                  AS summary_date,
    COUNT(*)                                                   AS total_tests,
    COUNT(DISTINCT patient_id)                                 AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                                  AS active_doctor_count,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)     AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)     AS critical_test_count
  FROM silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  d.hospital_id,
  h.hospital_name,
  d.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(d.total_tests AS INT)                                                 AS total_tests,
  CAST(d.unique_patients_tested AS INT)                                      AS unique_patients_tested,
  CAST(d.active_doctor_count AS INT)                                         AS active_doctor_count,
  CAST(d.abnormal_test_count AS INT)                                         AS abnormal_test_count,
  CAST(d.critical_test_count AS INT)                                         AS critical_test_count,
  CAST(ROUND(d.abnormal_test_count * 100.0 / d.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(d.critical_test_count * 100.0 / d.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(d.prev_day_total_tests AS INT)                                        AS prev_day_total_tests,
  CAST(
    CASE WHEN d.prev_day_total_tests IS NULL OR d.prev_day_total_tests = 0 THEN NULL
         ELSE ROUND((d.total_tests - d.prev_day_total_tests) * 100.0 / d.prev_day_total_tests, 2)
    END AS DECIMAL(6,2))                                                     AS test_volume_change_pct
FROM with_lag d
LEFT JOIN silver.hospitals h ON d.hospital_id = h.hospital_id
LEFT JOIN silver.calendar c  ON c.date_key = CAST(date_format(d.summary_date, 'yyyyMMdd') AS INT);

In [0]:
%sql
SELECT
  COUNT(*)                                         AS gold_rows,
  COUNT(DISTINCT hospital_id, summary_date)        AS distinct_hospital_days,
  SUM(total_tests)                                 AS total_tests,
  (SELECT COUNT(*) FROM silver.lab_results)        AS source_lab_results,
  SUM(CASE WHEN day_name IS NULL THEN 1 ELSE 0 END)      AS missing_calendar,
  SUM(CASE WHEN hospital_name IS NULL THEN 1 ELSE 0 END) AS missing_hospital
FROM gold.hospital_daily_summary